## O Efeito do Parcelamento no Ticket Médio
O cliente brasileiro tem uma forte cultura de compras a prazo. Esta análise isola os pagamentos feitos por **Cartão de Crédito** para entender como o número de prestações (parcelas) afeta diretamente o valor que o cliente está disposto a gastar (Ticket Médio).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="pastel")

# Carrega o Dataset Principal
df = pd.read_csv('../data/processed/olist_super_dataset.csv')

# Carrega a tabela extra de Pagamentos (necessária para esse gráfico específico)
pagamentos_raw = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

Sucesso! Dataset carregado com 111744 linhas.


In [ ]:
# 1. Juntar (Merge) as informações de pagamento com o nosso DataFrame principal
df_completo = df.merge(pagamentos_raw[['order_id', 'payment_type', 'payment_installments']], on='order_id', how='inner')

# 2. Filtrar apenas os pagamentos feitos por cartão de crédito
df_credito = df_completo[df_completo['payment_type'] == 'credit_card'].copy()

# 3. Agrupar pelo número de parcelas (prestações)
analise_parcelas = df_credito.groupby('payment_installments').agg(
    volume_pedidos=('order_id', 'count'),
    ticket_medio=('receita_liquida', 'mean')
).reset_index()

# 4. Limpar outliers (focar de 1 a 12) e forçar transformação para TEXTO ("1x", "2x"...)
analise_parcelas = analise_parcelas[(analise_parcelas['payment_installments'] >= 1) & 
                                    (analise_parcelas['payment_installments'] <= 12)].copy()

analise_parcelas['rotulo_parcela'] = analise_parcelas['payment_installments'].astype(str) + "x"

# 5. Plotagem do Gráfico Duplo
fig, ax1 = plt.subplots(figsize=(14, 6))

# Gráfico de Barras: Volume de Pedidos
color = '#3498db'
sns.barplot(x='rotulo_parcela', y='volume_pedidos', data=analise_parcelas, ax=ax1, color=color, alpha=0.7)
ax1.set_xlabel('Número de Prestações (Parcelas)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Volume de Pedidos', color=color, fontsize=12, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color)

# Gráfico de Linha: Ticket Médio perfeitamente alinhado
ax2 = ax1.twinx()
color = '#e67e22'
sns.lineplot(x='rotulo_parcela', y='ticket_medio', data=analise_parcelas, ax=ax2, color=color, marker='o', linewidth=3, markersize=10, sort=False)
ax2.set_ylabel('Ticket Médio (R$)', color=color, fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color)
ax2.grid(False) 

# Adicionar os valores do Ticket Médio por cima de cada ponto da linha
for i in range(len(analise_parcelas)):
    valor = analise_parcelas['ticket_medio'].iloc[i]
    ax2.annotate(f"R$ {valor:.0f}", 
                 (i, valor), 
                 textcoords="offset points", 
                 xytext=(0,15), 
                 ha='center', fontsize=10, fontweight='bold', color=color)
                 
plt.title('O Efeito do Parcelamento: Como dividir o pagamento aumenta o Ticket Médio', fontsize=15, pad=20, fontweight='bold')
plt.show()

As colunas 'payment_type' e/ou 'payment_installments' não estão presentes no DataFrame.


**Poder de Compra:** O gráfico revela uma correlação quase perfeita: **à medida que o número de prestações aumenta, o Ticket Médio dispara**. 
* Compras à vista (1x) ou em poucas prestações concentram o grande volume de pedidos, mas representam carrinhos de baixo valor (compras quotidianas ou de conveniência).
* No entanto, produtos como Eletrónicos ou Móveis, que custam muito mais do que a média, dependem vitalmente do parcelamento longo (10x a 12x) para caberem no orçamento mensal do consumidor.

**Ação Proposta para o Negócio:** Para aumentar a faturação geral (GMV), deve negociar taxas de adquirência mais baixas para poder oferecer e promover o "Parcelamento em até 12x sem juros" nas categorias de produtos mais caros. Subsidiar estes juros é um investimento que se paga pelo aumento drástico da conversão em compras de alto valor.

## Onde está o Volume?
Análise focada estritamente na distribuição da quantidade de vendas baseada no número de prestações escolhidas pelo cliente no pagamento via Cartão de Crédito.

In [ ]:
# 1. Agrupar e contar os pedidos
volume_parcelas = df_credito.groupby('payment_installments').agg(
    quantidade_vendas=('order_id', 'count')
).reset_index()

# 2. Limpar outliers (focar no padrão de 1 a 12 parcelas) e resetar o índice
volume_parcelas = volume_parcelas[(volume_parcelas['payment_installments'] >= 1) & 
                                  (volume_parcelas['payment_installments'] <= 12)].copy()
volume_parcelas = volume_parcelas.reset_index(drop=True)

# 3. Calcular a percentagem que cada parcela representa do total
total_vendas_credito = volume_parcelas['quantidade_vendas'].sum()
volume_parcelas['percentagem'] = (volume_parcelas['quantidade_vendas'] / total_vendas_credito) * 100

# 4. Criar o rótulo de texto (ex: "1x", "2x")
volume_parcelas['rotulo_parcela'] = volume_parcelas['payment_installments'].astype(str) + "x"

# 5. Plotagem do Gráfico
plt.figure(figsize=(14, 6))

# AQUI ESTÁ A CORREÇÃO: Adicionamos hue='rotulo_parcela' e legend=False
ax = sns.barplot(x='rotulo_parcela', y='quantidade_vendas', data=volume_parcelas, 
                 hue='rotulo_parcela', palette="viridis", alpha=0.9, legend=False)

plt.title('Distribuição do Volume de Vendas por Número de Prestações', fontsize=16, pad=20, fontweight='bold')
plt.xlabel('Número de Prestações (Parcelas)', fontsize=12, fontweight='bold')
plt.ylabel('Quantidade de Vendas (Pedidos)', fontsize=12, fontweight='bold')

# Adicionar os valores absolutos e a percentagem por cima de cada barra
for i, p in enumerate(ax.patches):
    altura = p.get_height()
    percentagem = volume_parcelas['percentagem'].iloc[i]
    ax.annotate(f"{int(altura)}\n({percentagem:.1f}%)", 
                (p.get_x() + p.get_width() / 2., altura), 
                ha='center', va='bottom', 
                xytext=(0, 5), 
                textcoords='offset points',
                fontsize=10, fontweight='bold', color='#2c3e50')

# Ajustar o limite do eixo Y para a label não cortar no topo
plt.ylim(0, volume_parcelas['quantidade_vendas'].max() * 1.15)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

**Apesar do parcelamento longo** (10x a 12x) ser crucial para aumentar o nosso Ticket Médio, como vimos no gráfico anterior, este novo gráfico mostra que o pagamento à vista no cartão de crédito (1x) ainda domina completamente o e-commerce, representando mais de 30% de todo o volume de transações de cartão. Isso indica que a maioria das compras na Olist são de itens de conveniência ou de menor valor, onde o cliente não sente necessidade de dividir a compra